# Case Study: Predicting Player Actions

**Chapter 5: Advanced Classification Methods - EXTRA**

## Learning Objectives

- Build a multi-class classifier for player decisions
- Predict whether a player will pass, dribble, or shoot
- Handle multi-class problems with tree-based models
- Extract tactical insights from model predictions

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsbombpy import sb
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## The Problem: Player Decision-Making

**Question:** Given a player's situation (position, opponents nearby, teammates available), what will they do?

**Possible Actions:**
- **Pass**: Move the ball to a teammate
- **Dribble**: Carry the ball forward
- **Shot**: Attempt to score

**Why This Matters:**
- Identify players who make optimal decisions
- Understand tactical patterns
- Predict opponent behavior

## Load Event Data

In [ ]:
# Load match events
matches = sb.matches(competition_id=16, season_id=4)
events = sb.events(match_id=22912)

print(f"Total events: {len(events)}")
print(f"\nEvent types:")
print(events['type'].value_counts().head(10))

## Filter for Relevant Actions

In [ ]:
# Filter for passes, dribbles, and shots
actions = events[events['type'].isin(['Pass', 'Dribble', 'Shot'])].copy()

print(f"\nTotal actions: {len(actions)}")
print(f"\nAction distribution:")
print(actions['type'].value_counts())

## Feature Engineering

In [ ]:
# Extract features
def extract_action_features(actions_df):
    df = actions_df.copy()
    
    # Position on field
    df['x'] = df['location'].apply(lambda loc: loc[0] if isinstance(loc, list) else None)
    df['y'] = df['location'].apply(lambda loc: loc[1] if isinstance(loc, list) else None)
    
    # Distance to goal
    df['distance_to_goal'] = df['location'].apply(
        lambda loc: np.sqrt((120 - loc[0])**2 + (40 - loc[1])**2) if isinstance(loc, list) else None
    )
    
    # Pressure (1 if under pressure, 0 otherwise)
    df['under_pressure'] = df['under_pressure'].fillna(False).astype(int)
    
    # Minute of match
    df['minute'] = df['minute'].fillna(0)
    
    # Action type (target variable)
    df['action'] = df['type']
    
    return df

actions = extract_action_features(actions)

print(f"\nFeature summary:")
print(actions[['x', 'y', 'distance_to_goal', 'under_pressure', 'minute']].describe())

## Prepare Data

In [ ]:
# Select features and target
feature_cols = ['x', 'y', 'distance_to_goal', 'under_pressure', 'minute']
X = actions[feature_cols].dropna()
y = actions.loc[X.index, 'action']

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

print(f"Training set: {len(X_train)} actions")
print(f"Test set: {len(X_test)} actions")
print(f"\nClass distribution in training:")
print(y_train.value_counts())

## Train Multi-Class Random Forest

In [ ]:
# Train Random Forest
rf_clf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
rf_clf.fit(X_train, y_train)

# Predictions
y_pred = rf_clf.predict(X_test)

# Evaluate
print("Classification Report:")
print(classification_report(y_test, y_pred))

## Confusion Matrix

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, y_pred, labels=['Pass', 'Dribble', 'Shot'])

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Pass', 'Dribble', 'Shot'],
            yticklabels=['Pass', 'Dribble', 'Shot'])
plt.ylabel('True Action')
plt.xlabel('Predicted Action')
plt.title('Confusion Matrix: Player Action Prediction')
plt.tight_layout()
plt.show()

## Feature Importance

In [ ]:
# Feature importance
importances = rf_clf.feature_importances_
feature_importance_df = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': importances
}).sort_values('Importance', ascending=False)

print("Feature Importances:")
print(feature_importance_df.to_string(index=False))

plt.figure(figsize=(10, 6))
plt.barh(feature_importance_df['Feature'], feature_importance_df['Importance'])
plt.xlabel('Importance')
plt.title('Feature Importance for Player Action Prediction')
plt.tight_layout()
plt.show()

## Tactical Insights

**From the model, we can learn:**

1. **Position matters most** (x, y coordinates)
   - Players in attacking third more likely to shoot
   - Players in midfield more likely to pass

2. **Distance to goal** influences decisions
   - Close to goal → more shots
   - Far from goal → more passes

3. **Pressure affects choices**
   - Under pressure → quicker decisions (passes or dribbles)
   - No pressure → more time to shoot

**Application:** Identify players who make suboptimal decisions (e.g., shooting from too far)

## Summary

In this case study, we:

1. ✅ Built a multi-class classifier for player actions
2. ✅ Predicted pass, dribble, or shot decisions
3. ✅ Analyzed feature importance
4. ✅ Extracted tactical insights

## Key Takeaways

- **Multi-class problems** are natural for tree-based models
- **Position and context** drive player decisions
- Models can **identify decision-making patterns**
- Useful for **player evaluation and tactical analysis**

## Practice Exercises

1. **Add More Features**: Include score differential, teammates nearby
2. **Player-Specific Models**: Build separate models for different positions
3. **Optimal Decisions**: Define "optimal" and measure player performance
4. **Temporal Patterns**: Analyze how decisions change over match time
5. **XGBoost Comparison**: Try XGBoost and compare with Random Forest